In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from datetime import datetime
import csv
import json

# 1. Configure scraping target and currency settings
base_url = "https://www.jumia.co.ke/smartphones/"

supported_currencies = {
    "KES": "Kenyan Shilling",
    "USD": "US Dollar",
    "GBP": "British Pound",
    "EUR": "Euro",

}

print("Supported currencies:", list(supported_currencies.keys()))
default_currency = input("Enter source currency (default KES): ").upper() or "KES"
target_currency = input("Enter target currency (default USD): ").upper() or "USD"

if default_currency not in supported_currencies or target_currency not in supported_currencies:
    print("Invalid currency! Defaulting to KES -> USD")
    default_currency = "KES"
    target_currency = "USD"

print(f"\nConverting from {default_currency} ({supported_currencies[default_currency]}) to {target_currency} ({supported_currencies[target_currency]})")

# 2. Fetch web page with error handling
soup = None
headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
try:
    response = requests.get(base_url, headers=headers, timeout=10)
    response.raise_for_status()
    print(f"\nSuccessfully fetched {base_url} (Status: {response.status_code})")
    soup = BeautifulSoup(response.content, "html.parser")
except requests.exceptions.ConnectionError:
    print("Connection Error: Could not connect to the website. Check your internet.")
except requests.exceptions.Timeout:
    print("Timeout Error: The server took too long to respond.")
except requests.exceptions.HTTPError as err:
    print(f"HTTP Error: {err}")

# 3. Parse and extract product names and prices
products = []

if soup:
    items = soup.select("article.prd._fb.col.c-prd")
    if not items:
        items = soup.select("article[class*='prd']")
    if not items:
        items = soup.select(".info")

    for item in items:
        try:
            name_tag = item.select_one("h3.name")
            if not name_tag:
                name_tag = item.select_one(".name")
            title = name_tag.get_text(strip=True) if name_tag else "N/A"

            price_tag = item.select_one(".prc")
            if not price_tag:
                price_tag = item.select_one(".price")
            raw_price = price_tag.get_text(strip=True) if price_tag else "N/A"

            if title != "N/A" and raw_price != "N/A":
                products.append({"title": title, "raw_price": raw_price})
                if len(products) >= 15:
                    break
        except Exception as e:
            print(f"Skipped a product due to error: {e}")
            continue

    print(f"Extracted {len(products)} products from the page.")
else:
    print("No page content to parse.")

# 4. Clean and normalize price data
cleaned_products = []

for product in products:
    raw = product["raw_price"]
    price_match = re.search(r"[\d,]+\.?\d*", raw.replace(" ", ""))
    if price_match:
        price_str = price_match.group().replace(",", "")
        try:
            price_float = float(price_str)
            cleaned_products.append({"title": product["title"], "original_price": price_float})
        except ValueError:
            print(f"Could not convert price: {raw}")
    else:
        print(f"No price found in: {raw}")

print(f"\n{len(cleaned_products)} products with valid prices.")

# 5. Fetch live currency exchange rates
exchange_rate = None

try:
    api_url = f"https://api.frankfurter.app/latest?from={default_currency}&to={target_currency}"
    api_response = requests.get(api_url, timeout=10)
    api_response.raise_for_status()
    rate_data = api_response.json()
    exchange_rate = rate_data["rates"][target_currency]
    print(f"Exchange Rate: 1 {default_currency} = {exchange_rate} {target_currency}")
    print(f"Rate Date: {rate_data['date']}")
except requests.exceptions.ConnectionError:
    print("Could not connect to the exchange rate API.")
except requests.exceptions.Timeout:
    print("Exchange rate API timed out.")
except requests.exceptions.HTTPError as err:
    print(f"HTTP Error from API: {err}")
except KeyError:
    print(f"Could not find rate for {default_currency} -> {target_currency}")

# 6. Convert prices to selected currency
if exchange_rate:
    for product in cleaned_products:
        product["converted_price"] = round(product["original_price"] * exchange_rate, 2)
    print(f"\nConverted {len(cleaned_products)} prices from {default_currency} to {target_currency}")
else:
    print("No exchange rate available for this conversion.")

# 7. Add timestamps
conversion_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
for product in cleaned_products:
    product["conversion_timestamp"] = conversion_time
print(f"Timestamp: {conversion_time}")

# 8. Display results in a formatted table
df = pd.DataFrame(cleaned_products)
df = df.rename(columns={
    "title": "Product Name",
    "original_price": f"Price ({default_currency})",
    "converted_price": f"Price ({target_currency})",
    "conversion_timestamp": "Converted At"
})

print(f"\n{'='*80}")
print(f"JUMIA KENYA - Product Prices ({default_currency} to {target_currency})")
print(f"Exchange Rate: 1 {default_currency} = {exchange_rate} {target_currency}")
print(f"{'='*80}")
display(df)

# 9. Save to CSV and JSON
df.to_csv("jumia_products_converted.csv", index=False)
print("Saved to jumia_products_converted.csv")

df.to_json("jumia_products_converted.json", orient="records", indent=2)
print("Saved to jumia_products_converted.json")

# 10. Plot original vs converted prices bar chart
        if conversion_success:
    plot_df = df.head(10)
    product_names = [name[:25] + "..." if len(name) > 25 else name for name in plot_df["Product Name"]]
    original_prices = plot_df[f"Price ({default_currency})"]
    converted_prices = plot_df[f"Price ({target_currency})"]

    x = np.arange(len(product_names))
    width = 0.35

    fig, ax1 = plt.subplots(figsize=(14, 6))

    bars1 = ax1.bar(x - width/2, original_prices, width, label=f"Original ({default_currency})", color="#2196F3")
    ax1.set_ylabel(f"Price ({default_currency})", color="#2196F3")
    ax1.tick_params(axis="y", labelcolor="#2196F3")

    ax2 = ax1.twinx()
    bars2 = ax2.bar(x + width/2, converted_prices, width, label=f"Converted ({target_currency})", color="#FF9800")
    ax2.set_ylabel(f"Price ({target_currency})", color="#FF9800")
    ax2.tick_params(axis="y", labelcolor="#FF9800")

    ax1.set_xlabel("Products")
    ax1.set_xticks(x)
    ax1.set_xticklabels(product_names, rotation=45, ha="right", fontsize=8)
    ax1.set_title(f"Jumia Kenya: Original ({default_currency}) vs Converted ({target_currency}) Prices\nRate: 1 {default_currency} = {exchange_rate} {target_currency}")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart — no conversion was performed.")